# Citi Bike Trip Data: where and when to rebalance bikes?
### SQL analysis with DuckDB and Jupyter

This notebook investigates one month of Citi Bike trips (Aug 31 to Sep 30, 2026, about 5 million trips). It answers 7 business questions in SQL and ends with a recommendation: which 5 stations should receive bikes overnight.

**How each question is organized:** the question (heading), the query with a comment explaining *why* it exists, and a conclusion that uses only numbers the query showed.

**Tables and views used**

| Name | What it is |
| --- | --- |
| `trips_raw` | the 5 CSV files loaded as they came, never changed |
| `trips_clean` | view over `trips_raw` without invalid trips |
| `stations` | one row per station id (id, name, average latitude and longitude) |
| `station_net_flow` | view: arrivals minus departures per station (month total) |
| `station_daily_flow` | view: arrivals minus departures per station and per day |

In [1]:
from pathlib import Path
import duckdb

BASE_DIR = Path().resolve().parent
db_path = BASE_DIR / "database" / "citibike.duckdb"

con = duckdb.connect(str(db_path))

%reload_ext sql
%sql con
%config SqlMagic.autolimit = 100
%config SqlMagic.displaylimit = 30

> `displaylimit = 30` matters: with the old limit of 15 rows, the weekend rows of the hourly table were hidden and the conclusion could not be checked.

## 1. Data quality and cleaning

Before answering anything, I check what is wrong with the raw data and decide what to remove. Every number in this section goes to the README.

### 1.1 What is wrong in `trips_raw`?

One query counts every problem I want to know about: trips that end before they start, trips under 60 seconds, trips over 24 hours, missing stations and duplicated trips.

In [2]:
%%sql
SELECT
    COUNT(*) AS total,
    COUNT(*) FILTER (WHERE ended_at <= started_at) AS inverted_or_zero,
    COUNT(*) FILTER (WHERE DATE_DIFF('second', started_at, ended_at) BETWEEN 1 AND 59) AS under_1_min,
    COUNT(*) FILTER (WHERE DATE_DIFF('second', started_at, ended_at) > 86400) AS over_24h,
    COUNT(*) FILTER (WHERE start_station_id IS NULL) AS no_start_station_id,
    COUNT(*) FILTER (WHERE end_station_id IS NULL) AS no_end_station_id,
    COUNT(*) - COUNT(DISTINCT ride_id) AS duplicated_rides
FROM trips_raw;

Running query in 'DuckDBPyConnection'

total,inverted_or_zero,under_1_min,over_24h,no_start_station_id,no_end_station_id,duplicated_rides
4982855,0,0,1070,2917,14576,0


> **Reading the result:** 4,982,855 trips, no duplicates, no inverted dates and **no trip under 60 seconds**. Only **1,070 trips (0.02%) last more than 24 hours**. The data provider probably already removed very short trips, so only the 24-hour rule changes anything.

### 1.2 What period does the data cover?

In [3]:
%%sql
SELECT
    MIN(started_at) AS first_start,
    MAX(started_at) AS last_start,
    MAX(ended_at) AS last_end
FROM trips_raw;

Running query in 'DuckDBPyConnection'

first_start,last_start,last_end
2026-08-31 00:50:32.105000,2026-09-30 23:58:16.821000,2026-09-30 23:59:59.555000


> **Reading the result:** the data starts on 2026-08-31 (a partial day) and ends on 2026-09-30. For any per-day analysis I use only Sep 1 to Sep 30, so a partial day does not distort averages.

### 1.3 Do station ids and station names agree?

A station can be identified by id or by name. The raw data has 14,576 trips with no `end_station_id` but only 13,455 with no `end_station_name`, so there are at least 1,121 trips with a name but no id. This query splits trips into the four possible cases, for the end station and then for the start station.

In [4]:
%%sql
SELECT
    COUNT(*) AS total,
    COUNT(*) FILTER (WHERE end_station_id IS NULL AND end_station_name IS NULL) AS id_null_name_null,
    COUNT(*) FILTER (WHERE end_station_id IS NULL AND end_station_name IS NOT NULL) AS id_null_name_present,
    COUNT(*) FILTER (WHERE end_station_id IS NOT NULL AND end_station_name IS NULL) AS id_present_name_null,
    COUNT(*) FILTER (WHERE end_station_id IS NOT NULL AND end_station_name IS NOT NULL) AS both_present
FROM trips_raw;

Running query in 'DuckDBPyConnection'

total,id_null_name_null,id_null_name_present,id_present_name_null,both_present
4982855,13455,1121,0,4968279


In [5]:
%%sql
SELECT
    COUNT(*) AS total,
    COUNT(*) FILTER (WHERE start_station_id IS NULL AND start_station_name IS NULL) AS id_null_name_null,
    COUNT(*) FILTER (WHERE start_station_id IS NULL AND start_station_name IS NOT NULL) AS id_null_name_present,
    COUNT(*) FILTER (WHERE start_station_id IS NOT NULL AND start_station_name IS NULL) AS id_present_name_null,
    COUNT(*) FILTER (WHERE start_station_id IS NOT NULL AND start_station_name IS NOT NULL) AS both_present
FROM trips_raw;

Running query in 'DuckDBPyConnection'

total,id_null_name_null,id_null_name_present,id_present_name_null,both_present
4982855,2917,0,0,4979938


Checking a sample of the trips that have a name but no id, to understand what they are:

In [6]:
%%sql
SELECT
    end_station_name,
    rideable_type,
    COUNT(*) AS trips
FROM trips_raw
WHERE end_station_id IS NULL
  AND end_station_name IS NOT NULL
GROUP BY end_station_name, rideable_type
ORDER BY trips DESC
LIMIT 20;

Running query in 'DuckDBPyConnection'

end_station_name,rideable_type,trips
E 33 St & 1 Ave,electric_bike,116
Amsterdam Ave & W 73 St,electric_bike,64
Hancock St & Bedford Ave,electric_bike,41
E 85 St & 3 Ave,electric_bike,34
W 116 St & Broadway,electric_bike,23
Allen St & Hester St,electric_bike,22
St James Pl & Oliver St,electric_bike,21
Eastern Pkwy & Franklin Ave (NE Corner),electric_bike,19
W 43 St & 6 Ave,electric_bike,18
Broadway & E 21 St,electric_bike,18


Does each station name belong to one id, and each id to one name? If not, grouping by name would merge different stations (or split one station in two).

In [ ]:
%%sql
WITH pairs AS (
    SELECT DISTINCT
        end_station_name AS station_name,
        end_station_id AS station_id
    FROM trips_raw
    WHERE end_station_name IS NOT NULL
      AND end_station_id IS NOT NULL
)
SELECT
    (SELECT COUNT(*) FROM (SELECT station_name FROM pairs GROUP BY station_name HAVING COUNT(*) > 1)) AS names_with_several_ids,
    (SELECT COUNT(*) FROM (SELECT station_id FROM pairs GROUP BY station_id HAVING COUNT(*) > 1)) AS ids_with_several_names;

Running query in 'DuckDBPyConnection'

names_with_several_ids,ids_with_several_names
62,1


> **Decision:**
> - Result of the 4-case split: 1,121 trips have an end station name but no end id, and 0 trips have a start station name without a start id. No trip has an id without a name.
> - Result of the name/id uniqueness check: 62 station names are linked to more than one id, and 1 id is linked to more than one name.
> - From here on, station questions group by **station name**, because the names are present in the trips where the id is missing. The cost is that a few stations may be merged or split. I did not check whether the 62 names with several ids are the same physical place, so this is listed as a limitation in the README.

### 1.4 The clean view

`trips_clean` is the only table used from now on. The raw data stays untouched in `trips_raw`.

In [8]:
%%sql
/*
trips_clean: filtered view over trips_raw, used by all analysis queries.

Removes:
- ended_at <= started_at: a trip cannot end before it starts, so these are
  system or recording errors. (0 rows in this dataset)
- duration < 60 seconds: usually false starts (bike undocked and returned
  right away). (0 rows in this dataset, the provider already removed them)
- duration > 24 hours (86,400 s): usually lost, forgotten or unreturned bikes.
  (1,070 rows)

The raw data stays untouched in trips_raw.
Station nulls are NOT removed here: they are only excluded in the questions
about stations, so the questions about hours and riders still use every trip.
*/
CREATE OR REPLACE VIEW trips_clean AS (
    SELECT
        * EXCLUDE (filename),
        DATE_DIFF('second', started_at, ended_at) AS duration_seconds
    FROM trips_raw
    WHERE ended_at > started_at
        AND DATE_DIFF('second', started_at, ended_at) >= 60
        AND DATE_DIFF('second', started_at, ended_at) <= 86400
);

Running query in 'DuckDBPyConnection'

Count


How many rows did each rule remove? This is the table that goes in the README.

In [9]:
%%sql
SELECT
    COUNT(*) AS raw_rows,
    COUNT(*) FILTER (WHERE ended_at <= started_at) AS removed_end_before_start,
    COUNT(*) FILTER (WHERE DATE_DIFF('second', started_at, ended_at) BETWEEN 1 AND 59) AS removed_under_60s,
    COUNT(*) FILTER (WHERE DATE_DIFF('second', started_at, ended_at) > 86400) AS removed_over_24h,
    (SELECT COUNT(*) FROM trips_clean) AS clean_rows,
    COUNT(*) - (SELECT COUNT(*) FROM trips_clean) AS total_removed
FROM trips_raw;

Running query in 'DuckDBPyConnection'

raw_rows,removed_end_before_start,removed_under_60s,removed_over_24h,clean_rows,total_removed
4982855,0,0,1070,4981785,1070


Missing stations in the clean data (these are **trips**, not stations). Which bike type has them?

In [10]:
%%sql
SELECT
    rideable_type,
    COUNT(*) FILTER (WHERE start_station_name IS NULL) AS trips_without_start_station,
    COUNT(*) FILTER (WHERE end_station_name IS NULL) AS trips_without_end_station
FROM trips_clean
GROUP BY rideable_type;

Running query in 'DuckDBPyConnection'

rideable_type,trips_without_start_station,trips_without_end_station
classic_bike,0,66
electric_bike,2917,12338


> **Conclusion:** the cleaning removed 1,070 trips (0.02% of 4,982,855), all because they lasted more than 24 hours. Missing stations are almost all electric bikes (2,917 trips without a start station and 12,338 without an end station, against 0 and 66 for classic bikes), probably because electric bikes can be locked outside a dock. These trips are kept for the questions about hours, riders and duration, and excluded from the questions about stations.

### 1.5 The `stations` table

One row per station id, with the most common name and the average coordinates (GPS changes a little from trip to trip, so a plain `SELECT DISTINCT` would return several rows per station).

In [11]:
%%sql
CREATE OR REPLACE TABLE stations AS
WITH all_points AS (
    SELECT
        start_station_id AS station_id,
        start_station_name AS station_name,
        start_lat AS lat,
        start_lng AS lng
    FROM trips_raw
    UNION ALL
    SELECT end_station_id, end_station_name, end_lat, end_lng
    FROM trips_raw
)
SELECT
    station_id,
    MODE(station_name) AS station_name,
    AVG(lat) AS lat,
    AVG(lng) AS lng,
    COUNT(*) AS trip_ends
FROM all_points
WHERE station_id IS NOT NULL
GROUP BY station_id;

Running query in 'DuckDBPyConnection'

Count
2501


In [12]:
%%sql
SELECT COUNT(*) AS stations, COUNT(DISTINCT station_id) AS unique_ids FROM stations;

Running query in 'DuckDBPyConnection'

stations,unique_ids
2501,2501


## Question 1. For each station, what is the net flow (bikes arriving minus bikes departing)? Which stations are consistently bike deficits and which are bike surpluses?

**What we are looking for:** `net_flow = arrivals - departures`. A **negative** number means the station loses bikes (it needs bikes brought in). A **positive** number means bikes pile up. "Consistently" means the sign repeats on most days, so I look at the month total and then at each day.

### Q1.1 Month total per station

Departures and arrivals are counted in two separate CTEs and joined. Station nulls are excluded in both CTEs, because a "station" with no name is not a station. A `FULL JOIN` keeps stations that only appear as departure or only as arrival. The view is saved so Question 7 can reuse it.

In [13]:
%%sql
CREATE OR REPLACE VIEW station_net_flow AS (
    WITH departing AS (
        SELECT 
            start_station_name AS station, 
            COUNT(*) AS bikes_departing
        FROM trips_clean
        WHERE start_station_name IS NOT NULL
        GROUP BY start_station_name
    ),
    arriving AS (
        SELECT 
            end_station_name AS station, 
            COUNT(*) AS bikes_arriving
        FROM trips_clean
        WHERE end_station_name IS NOT NULL
        GROUP BY end_station_name
    )
    SELECT
        COALESCE(d.station, a.station) AS station,
        COALESCE(d.bikes_departing, 0) AS bikes_departing,
        COALESCE(a.bikes_arriving, 0) AS bikes_arriving,
        COALESCE(a.bikes_arriving, 0) - COALESCE(d.bikes_departing, 0) AS net_flow,
        ROUND(
            100.0 * (COALESCE(a.bikes_arriving, 0) - COALESCE(d.bikes_departing, 0))
            / (COALESCE(a.bikes_arriving, 0) + COALESCE(d.bikes_departing, 0)), 1
        ) AS net_flow_pct_of_volume,
        CASE
            WHEN COALESCE(a.bikes_arriving, 0) < COALESCE(d.bikes_departing, 0) THEN 'bike deficit'
            WHEN COALESCE(a.bikes_arriving, 0) > COALESCE(d.bikes_departing, 0) THEN 'bike surplus'
            ELSE 'balanced'
        END AS classification
    FROM departing d
    FULL JOIN arriving a ON d.station = a.station
);

Running query in 'DuckDBPyConnection'

Count


Top 10 deficits (more departures than arrivals):

In [14]:
%%sql
SELECT * FROM station_net_flow ORDER BY net_flow ASC LIMIT 10;

Running query in 'DuckDBPyConnection'

station,bikes_departing,bikes_arriving,net_flow,net_flow_pct_of_volume,classification
E 39 St & Lexington Ave,7489,6849,-640,-4.5,bike deficit
E 33 St & Park Ave,4165,3552,-613,-7.9,bike deficit
Willoughby Ave & Hall St,5593,5100,-493,-4.6,bike deficit
E 39 St & 2 Ave,5768,5276,-492,-4.5,bike deficit
E 51 St & 2 Ave,5726,5248,-478,-4.4,bike deficit
W 45 St & 8 Ave,8409,8058,-351,-2.1,bike deficit
E 35 St & Madison Ave,5103,4757,-346,-3.5,bike deficit
6 Ave & W 34 St,8856,8531,-325,-1.9,bike deficit
46 St & Queens Blvd,1810,1489,-321,-9.7,bike deficit
8 Ave & W 33 St,8800,8485,-315,-1.8,bike deficit


Top 10 surpluses (more arrivals than departures):

In [15]:
%%sql
SELECT * FROM station_net_flow ORDER BY net_flow DESC LIMIT 10;

Running query in 'DuckDBPyConnection'

station,bikes_departing,bikes_arriving,net_flow,net_flow_pct_of_volume,classification
Greenwich St & W Houston St,7770,8375,605,3.7,bike surplus
Pioneer St & Van Brunt St,2091,2360,269,6.0,bike surplus
E 93 St & 2 Ave,2536,2804,268,5.0,bike surplus
W 33 St & 10 Ave,5147,5357,210,2.0,bike surplus
Broadway & E 21 St,11170,11378,208,0.9,bike surplus
Columbia St & Lorraine St,479,675,196,17.0,bike surplus
Degraw St & 3 Ave,1863,2059,196,5.0,bike surplus
Murray St & Greenwich St,4686,4876,190,2.0,bike surplus
Kent Ave & N 7 St,6903,7091,188,1.3,bike surplus
Coffey St & Conover St,1232,1397,165,6.3,bike surplus


### Q1.2 Is the pattern consistent day by day?

I turn every trip into a **movement**: -1 at the departure station on the start date, +1 at the arrival station on the end date. Summing movements by station and day gives the daily net flow without any join. Only Sep 1 to Sep 30 is used (Aug 31 is a partial day). Stations with fewer than 20 active days are ignored because their pattern is not reliable.

In [32]:
%%sql
CREATE OR REPLACE VIEW station_daily_flow AS (
    WITH movements AS (
        SELECT 
            start_station_name AS station, 
            CAST(started_at AS DATE) AS day, 
            -1 AS flow
        FROM trips_clean
        WHERE start_station_name IS NOT NULL
        UNION ALL
        SELECT 
            end_station_name, 
            CAST(ended_at AS DATE), 1
        FROM trips_clean
        WHERE end_station_name IS NOT NULL
    )
    SELECT
        station,
        day,
        SUM(flow) AS net_flow,
        COUNT(*) AS movements
    FROM movements
    WHERE day BETWEEN DATE '2026-09-01' AND DATE '2026-09-30'
    GROUP BY station, day
);

Running query in 'DuckDBPyConnection'

Count


In [17]:
%%sql
SELECT
    station,
    COUNT(*) AS active_days,
    COUNT(*) FILTER (WHERE net_flow < 0) AS deficit_days,
    COUNT(*) FILTER (WHERE net_flow > 0) AS surplus_days,
    ROUND(100.0 * COUNT(*) FILTER (WHERE net_flow < 0) / COUNT(*), 1) AS pct_deficit_days,
    SUM(net_flow) AS month_net_flow
FROM station_daily_flow
GROUP BY station
HAVING COUNT(*) >= 20
ORDER BY month_net_flow ASC
LIMIT 15;

Running query in 'DuckDBPyConnection'

station,active_days,deficit_days,surplus_days,pct_deficit_days,month_net_flow
E 39 St & Lexington Ave,30,25,5,83.3,-640
E 33 St & Park Ave,30,30,0,100.0,-613
Willoughby Ave & Hall St,30,25,3,83.3,-493
E 39 St & 2 Ave,27,24,2,88.9,-492
E 51 St & 2 Ave,30,26,4,86.7,-478
W 45 St & 8 Ave,30,23,6,76.7,-348
E 35 St & Madison Ave,30,23,5,76.7,-345
6 Ave & W 34 St,30,25,4,83.3,-323
46 St & Queens Blvd,30,26,2,86.7,-321
8 Ave & W 33 St,30,20,9,66.7,-314


How many stations are consistent? I call a station a **consistent deficit** when it loses bikes on at least 80% of its active days, and a **consistent surplus** when it gains bikes on at least 80%. The 80% threshold is my choice, so I state it in the conclusion.

In [18]:
%%sql
WITH station_pattern AS (
    SELECT
        station,
        100.0 * COUNT(*) FILTER (WHERE net_flow < 0) / COUNT(*) AS pct_deficit_days,
        100.0 * COUNT(*) FILTER (WHERE net_flow > 0) / COUNT(*) AS pct_surplus_days
    FROM station_daily_flow
    GROUP BY station
    HAVING COUNT(*) >= 20
)
SELECT
    CASE
        WHEN pct_deficit_days >= 80 THEN 'consistent deficit'
        WHEN pct_surplus_days >= 80 THEN 'consistent surplus'
        ELSE 'mixed'
    END AS pattern,
    COUNT(*) AS stations
FROM station_pattern
GROUP BY pattern
ORDER BY stations DESC;

Running query in 'DuckDBPyConnection'

pattern,stations
mixed,2297
consistent deficit,13
consistent surplus,3


> **Conclusion:** over the month, the stations that lose the most bikes are E 39 St & Lexington Ave (-640), E 33 St & Park Ave (-613), Willoughby Ave & Hall St (-493), E 39 St & 2 Ave (-492) and E 51 St & 2 Ave (-478).
> - Among 2,313 stations with at least 20 active days, 13 are consistent deficits (a deficit on at least 80% of the days), 3 are consistent surpluses and 2,297 are mixed.
> - All five stations above are consistent deficits: they lose bikes on 83.3% to 100% of their active days.
> - Limitation: the month total is a simple difference. It does not say *when* in the day the bikes are missing (see Question 7).

## Question 2. What are the busiest hours for bike departures, overall and weekday vs weekend?

**What we are looking for:** the hours with the most departures, and whether the shape of the day changes on weekends.

### Q2.1 Overall: top 6 hours

I group by hour of `started_at` and show each hour's share of all departures.

In [19]:
%%sql
SELECT
    EXTRACT(HOUR FROM started_at) AS hour_of_day,
    COUNT(*) AS departures,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_of_all_departures
FROM trips_clean
GROUP BY hour_of_day
ORDER BY departures DESC
LIMIT 6;

Running query in 'DuckDBPyConnection'

hour_of_day,departures,pct_of_all_departures
17,484123,9.7
18,453018,9.1
16,371007,7.4
19,345305,6.9
15,318174,6.4
8,317936,6.4


### Q2.2 Weekday vs weekend: average departures **per day**

September 2026 has 22 weekdays and 8 weekend days, so comparing totals would favor weekdays only because there are more of them. I divide each hour's total by the number of days of that type. Only Sep 1 to Sep 30 is used.

In [33]:
%%sql
WITH trips_by_day_hour AS (
    SELECT
        CAST(started_at AS DATE) AS day,
        CASE WHEN EXTRACT(ISODOW FROM started_at) <= 5 THEN 'weekday' ELSE 'weekend' END AS day_type,
        EXTRACT(HOUR FROM started_at) AS hour_of_day,
        COUNT(*) AS trips
    FROM trips_clean
    WHERE started_at >= TIMESTAMP '2026-09-01'
    GROUP BY 1, 2, 3
),
day_counts AS (
    SELECT day_type, COUNT(DISTINCT day) AS n_days
    FROM trips_by_day_hour
    GROUP BY day_type
)
SELECT
    hour_of_day,
    ROUND(SUM(trips) FILTER (WHERE day_type = 'weekday')
          / (SELECT n_days FROM day_counts WHERE day_type = 'weekday'), 0) AS avg_departures_weekday,
    ROUND(SUM(trips) FILTER (WHERE day_type = 'weekend')
          / (SELECT n_days FROM day_counts WHERE day_type = 'weekend'), 0) AS avg_departures_weekend
FROM trips_by_day_hour
GROUP BY hour_of_day
ORDER BY hour_of_day;

Running query in 'DuckDBPyConnection'

hour_of_day,avg_departures_weekday,avg_departures_weekend
0,1897.0,3928.0
1,995.0,2736.0
2,576.0,1903.0
3,376.0,1223.0
4,469.0,757.0
5,1437.0,542.0
6,4112.0,1020.0
7,8640.0,1724.0
8,13422.0,2832.0
9,9981.0,4452.0


> **Conclusion:** overall, the busiest hours are 17h (484,123 departures, 9.7% of the month), 18h (453,018) and 16h (371,007). On weekdays there are two peaks, about 13,400 departures per day at 8h and about 18,400 at 17h, which is the shape of a commute pattern. On weekends there are no rush-hour peaks: departures stay on a plateau of about 8,800 to 10,000 per hour between 12h and 18h, with the maximum at 15h (10,007).
> Limitation: Monday Sep 7 (Labor Day) is counted as a weekday although it behaves like a holiday.

## Question 3. Which 10 stations have the highest total trip volume (starts + ends)?

**What we are looking for:** the busiest stations when both departures and arrivals are counted. I stack departures and arrivals with `UNION ALL` (keeps every row) and count by station.

In [21]:
%%sql
WITH station_events AS (
    SELECT start_station_name AS station, 'start' AS direction
    FROM trips_clean
    WHERE start_station_name IS NOT NULL
    UNION ALL
    SELECT end_station_name, 'end'
    FROM trips_clean
    WHERE end_station_name IS NOT NULL
)
SELECT
    station,
    COUNT(*) AS total_trips_volume,
    COUNT(*) FILTER (WHERE direction = 'start') AS starts,
    COUNT(*) FILTER (WHERE direction = 'end') AS ends
FROM station_events
GROUP BY station
ORDER BY total_trips_volume DESC
LIMIT 10;

Running query in 'DuckDBPyConnection'

station,total_trips_volume,starts,ends
W 21 St & 6 Ave,34636,17304,17332
Pier 61 at Chelsea Piers,33762,16870,16892
E 17 St & Broadway,30583,15252,15331
W 31 St & 7 Ave,30482,15313,15169
9 Ave & W 33 St,30019,14981,15038
Broadway & W 58 St,29602,14844,14758
Park Ave & E 42 St,29176,14529,14647
University Pl & E 14 St,27546,13755,13791
Broadway & E 14 St,27242,13613,13629
Broadway & W 25 St,26512,13240,13272


> **Conclusion:** the busiest station is W 21 St & 6 Ave (34,636 starts plus ends), followed by Pier 61 at Chelsea Piers (33,762) and E 17 St & Broadway (30,583). A round trip counts twice (once as start and once as end), which follows the definition in the question. None of the 15 stations with the biggest deficit in Question 1 is in this top 10, so a station with a large net flow is not necessarily a high-volume station.

## Question 4. Is there a difference in trip duration or patterns between members and casual riders?

**What we are looking for:** differences in how many trips, how long, which bike and when. The mean is pulled up by long trips, so I also show the **median**.

### Q4.1 Size and duration

In [22]:
%%sql
SELECT
    member_casual,
    COUNT(*) AS trips,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_of_trips,
    ROUND(AVG(duration_seconds) / 60, 1) AS avg_minutes,
    ROUND(MEDIAN(duration_seconds) / 60, 1) AS median_minutes
FROM trips_clean
GROUP BY member_casual
ORDER BY trips DESC;

Running query in 'DuckDBPyConnection'

member_casual,trips,pct_of_trips,avg_minutes,median_minutes
member,4047045,81.2,11.9,8.9
casual,934740,18.8,18.5,12.7


### Q4.2 Bike type inside each group

`pct_within_group` divides by the total of the group, not by all trips, so the two groups can be compared fairly.

In [23]:
%%sql
SELECT
    member_casual,
    rideable_type,
    COUNT(*) AS trips,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY member_casual), 1) AS pct_within_group,
    ROUND(AVG(duration_seconds) / 60, 1) AS avg_minutes,
    ROUND(MEDIAN(duration_seconds) / 60, 1) AS median_minutes
FROM trips_clean
GROUP BY member_casual, rideable_type
ORDER BY member_casual, trips DESC;

Running query in 'DuckDBPyConnection'

member_casual,rideable_type,trips,pct_within_group,avg_minutes,median_minutes
casual,electric_bike,719535,77.0,17.4,12.1
casual,classic_bike,215205,23.0,22.0,15.2
member,electric_bike,2899274,71.6,12.0,9.2
member,classic_bike,1147771,28.4,11.7,8.1


### Q4.3 When do they ride?

Share of each group's trips on weekends and in commute hours (7 to 9h and 16 to 19h, my definition).

In [24]:
%%sql
SELECT
    member_casual,
    ROUND(100.0 * COUNT(*) FILTER (WHERE EXTRACT(ISODOW FROM started_at) >= 6) / COUNT(*), 1) AS pct_weekend_trips,
    ROUND(100.0 * COUNT(*) FILTER (WHERE EXTRACT(HOUR FROM started_at) IN (7, 8, 9, 16, 17, 18, 19)) / COUNT(*), 1) AS pct_commute_hour_trips
FROM trips_clean
GROUP BY member_casual;

Running query in 'DuckDBPyConnection'

member_casual,pct_weekend_trips,pct_commute_hour_trips
member,18.8,50.2
casual,28.3,42.5


> **Conclusion:** members make 81.2% of the trips (4,047,045) and casual riders 18.8% (934,740). Casual trips are longer: 18.5 minutes on average against 11.9 for members. Both groups prefer electric bikes (71.6% of member trips and 77.0% of casual trips). Casual riders also keep longer rides on classic bikes (22.0 minutes), while members ride about the same on both bike types (11.7 and 12.0). `Members are more commute-oriented, with 50.2% of trips occurring during commute hours and only 18.8% on weekends, whereas casual riders have fewer commute-hour trips (42.5%) and a higher weekend share (28.3%).`.

## Question 5. What is the average trip duration, and are there suspicious outliers (0 seconds, several days) that suggest bad data?

**What we are looking for:** the duration of trips **before** cleaning (to see the problem) and **after** cleaning (to see what is left). Checking outliers on `trips_clean` would always return zero, because the view already removed them.

In [25]:
%%sql
WITH durations AS (
    SELECT 'raw' AS dataset, DATE_DIFF('second', started_at, ended_at) AS duration_seconds
    FROM trips_raw
    UNION ALL
    SELECT 'clean', duration_seconds
    FROM trips_clean
)
SELECT
    dataset,
    COUNT(*) AS trips,
    ROUND(MIN(duration_seconds) / 60.0, 1) AS min_minutes,
    ROUND(AVG(duration_seconds) / 60.0, 1) AS avg_minutes,
    ROUND(QUANTILE_CONT(duration_seconds, 0.5) / 60.0, 1) AS median_minutes,
    ROUND(QUANTILE_CONT(duration_seconds, 0.95) / 60.0, 1) AS p95_minutes,
    ROUND(QUANTILE_CONT(duration_seconds, 0.99) / 60.0, 1) AS p99_minutes,
    ROUND(MAX(duration_seconds) / 3600.0, 1) AS max_hours
FROM durations
GROUP BY dataset
ORDER BY dataset DESC;

Running query in 'DuckDBPyConnection'

dataset,trips,min_minutes,avg_minutes,median_minutes,p95_minutes,p99_minutes,max_hours
raw,4982855,1.0,13.4,9.5,33.6,58.7,25.0
clean,4981785,1.0,13.1,9.5,33.5,58.2,24.0


How many long trips are still in the clean data? This shows whether the 24-hour limit is reasonable.

In [26]:
%%sql
SELECT
    COUNT(*) FILTER (WHERE duration_seconds > 7200) AS trips_over_2h,
    COUNT(*) FILTER (WHERE duration_seconds > 21600) AS trips_over_6h,
    COUNT(*) FILTER (WHERE duration_seconds > 43200) AS trips_over_12h
FROM trips_clean;

Running query in 'DuckDBPyConnection'

trips_over_2h,trips_over_6h,trips_over_12h
7744,1614,569


> **Conclusion:** the average clean trip lasts 13.1 minutes (median 9.5). In the raw data, no trip is shorter than 60 seconds or ends before it starts, and the longest trip lasts about 25 hours, so there are no multi-day trips. The 1,070 trips (0.02%) longer than 24 hours were removed, which moved the average from 13.4 to 13.1 minutes and left the median unchanged. In the clean data, p95 is 33.5 minutes and p99 is 58.2 minutes. Some long trips remain below the limit (7,744 above 2 hours, 1,614 above 6 hours and 569 above 12 hours), so outliers are rare, not absent.

## Question 6. Which start-end station pairs are the most common routes? Is it a commuter pattern (same stations, weekday mornings) or leisure (weekend, varied stations)?

**What we are looking for:** the most common routes, and for each one *when* it happens and *who* rides it.

### Q6.1 Top 20 routes

`is_round_trip` is true when the trip starts and ends at the same station.

In [27]:
%%sql
SELECT
    start_station_name,
    end_station_name,
    start_station_name = end_station_name AS is_round_trip,
    COUNT(*) AS trips
FROM trips_clean
WHERE start_station_name IS NOT NULL
  AND end_station_name IS NOT NULL
GROUP BY start_station_name, end_station_name
ORDER BY trips DESC
LIMIT 20;

Running query in 'DuckDBPyConnection'

start_station_name,end_station_name,is_round_trip,trips
Central Park S & 6 Ave,Central Park S & 6 Ave,True,2138
7 Ave & Central Park South,7 Ave & Central Park South,True,1310
55 Ave & Center Blvd,Vernon Blvd & 50 Ave,False,987
Vernon Blvd & 50 Ave,55 Ave & Center Blvd,False,917
Roosevelt Island Tramway,Roosevelt Island Tramway,True,846
S 4 St & Wythe Ave,N 6 St & Bedford Ave,False,836
Soissons Landing,Soissons Landing,True,830
Eastern Pkwy & Kingston Ave,Eastern Pkwy & Kingston Ave,True,814
N 6 St & Bedford Ave,S 2 St & Kent Ave,False,810
N 6 St & Bedford Ave,S 4 St & Wythe Ave,False,781


### Q6.2 How much of all traffic are round trips?

In [28]:
%%sql
SELECT
    start_station_name = end_station_name AS is_round_trip,
    COUNT(*) AS trips,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_of_trips
FROM trips_clean
WHERE start_station_name IS NOT NULL
  AND end_station_name IS NOT NULL
GROUP BY is_round_trip;

Running query in 'DuckDBPyConnection'

is_round_trip,trips,pct_of_trips
False,4865230,98.0
True,101559,2.0


### Q6.3 Profile of each top route

To call a route "commuter" I need evidence: a high share of weekday trips, a high share in commute hours and a high share of members. First the baseline (all trips), then the top 20 routes to compare against it.

In [29]:
%%sql
SELECT
    'ALL TRIPS (baseline)' AS route,
    COUNT(*) AS trips,
    ROUND(100.0 * COUNT(*) FILTER (WHERE EXTRACT(ISODOW FROM started_at) <= 5) / COUNT(*), 1) AS pct_weekday,
    ROUND(100.0 * COUNT(*) FILTER (WHERE EXTRACT(HOUR FROM started_at) IN (7, 8, 9, 16, 17, 18, 19)) / COUNT(*), 1) AS pct_commute_hours,
    ROUND(100.0 * COUNT(*) FILTER (WHERE member_casual = 'member') / COUNT(*), 1) AS pct_member
FROM trips_clean;

Running query in 'DuckDBPyConnection'

route,trips,pct_weekday,pct_commute_hours,pct_member
ALL TRIPS (baseline),4981785,79.4,48.8,81.2


In [30]:
%%sql
WITH top_routes AS (
    SELECT
        start_station_name AS start_station,
        end_station_name AS end_station,
        COUNT(*) AS trips
    FROM trips_clean
    WHERE start_station_name IS NOT NULL
      AND end_station_name IS NOT NULL
    GROUP BY start_station_name, end_station_name
    ORDER BY trips DESC
    LIMIT 20
)
SELECT
    r.start_station,
    r.end_station,
    r.start_station = r.end_station AS is_round_trip,
    r.trips,
    ROUND(100.0 * COUNT(*) FILTER (WHERE EXTRACT(ISODOW FROM t.started_at) <= 5) / COUNT(*), 1) AS pct_weekday,
    ROUND(100.0 * COUNT(*) FILTER (WHERE EXTRACT(HOUR FROM t.started_at) IN (7, 8, 9, 16, 17, 18, 19)) / COUNT(*), 1) AS pct_commute_hours,
    ROUND(100.0 * COUNT(*) FILTER (WHERE t.member_casual = 'member') / COUNT(*), 1) AS pct_member
FROM top_routes r
JOIN trips_clean t
    ON t.start_station_name = r.start_station
   AND t.end_station_name = r.end_station
GROUP BY r.start_station, r.end_station, r.trips
ORDER BY r.trips DESC;

Running query in 'DuckDBPyConnection'

start_station,end_station,is_round_trip,trips,pct_weekday,pct_commute_hours,pct_member
Central Park S & 6 Ave,Central Park S & 6 Ave,True,2138,64.6,37.7,12.3
7 Ave & Central Park South,7 Ave & Central Park South,True,1310,67.4,39.1,9.6
55 Ave & Center Blvd,Vernon Blvd & 50 Ave,False,987,77.4,49.2,92.8
Vernon Blvd & 50 Ave,55 Ave & Center Blvd,False,917,80.9,53.2,94.5
Roosevelt Island Tramway,Roosevelt Island Tramway,True,846,64.1,47.5,19.5
S 4 St & Wythe Ave,N 6 St & Bedford Ave,False,836,83.3,62.4,87.9
Soissons Landing,Soissons Landing,True,830,59.5,30.6,22.9
Eastern Pkwy & Kingston Ave,Eastern Pkwy & Kingston Ave,True,814,94.3,29.6,93.9
N 6 St & Bedford Ave,S 2 St & Kent Ave,False,810,88.5,62.6,91.5
N 6 St & Bedford Ave,S 4 St & Wythe Ave,False,781,86.2,52.4,86.9


> **Conclusion:** the top route is a round trip at Central Park S & 6 Ave (2,138 trips), and 1,874 of them (87.7%) are casual riders, a leisure pattern around the park. On the other hand, Vernon Blvd & 50 Ave and 55 Ave & Center Blvd are connected in both directions (987 and 917 trips) and about 93% to 95% of the trips are members, which fits a regular, commute-like use. `Compared with the baseline, round trips are generally more leisure-oriented (lower weekday and commute-hour shares, higher casual-rider share), while the top member-heavy routes are consistent with commuting behavior.`.

## Question 7. Which 5 stations would you recommend for overnight rebalancing (moving bikes in), and what data backs it?

**What we are looking for:** stations that run short of bikes in the morning. Three pieces of evidence, from the previous questions:
1. **Month net flow** is negative (more departures than arrivals).
2. It is **consistent**: deficit on at least 70% of the days (Q1.2).
3. It **matters**: at least 2,000 movements in the month (a -300 at a station with 1,800 trips is a much bigger problem than -300 at one with 10,000, and tiny stations are noisy) and many **weekday morning departures** (6 to 9h), which is when the empty docks hurt customers.

The thresholds (70% and 2,000) are my choices. The list shows 10 candidates, and the recommendation takes the top 5.

In [31]:
%%sql
WITH station_summary AS (
    SELECT
        station,
        COUNT(*) AS active_days,
        ROUND(100.0 * COUNT(*) FILTER (WHERE net_flow < 0) / COUNT(*), 1) AS pct_deficit_days,
        SUM(net_flow) AS month_net_flow,
        SUM(movements) AS total_movements
    FROM station_daily_flow
    GROUP BY station
    HAVING COUNT(*) >= 20
),
morning AS (
    SELECT
        start_station_name AS station,
        COUNT(*) AS weekday_morning_departures
    FROM trips_clean
    WHERE start_station_name IS NOT NULL
      AND EXTRACT(ISODOW FROM started_at) <= 5
      AND EXTRACT(HOUR FROM started_at) BETWEEN 6 AND 9
    GROUP BY start_station_name
)
SELECT
    s.station,
    s.month_net_flow,
    ROUND(100.0 * s.month_net_flow / s.total_movements, 1) AS net_flow_pct_of_volume,
    s.pct_deficit_days,
    s.total_movements,
    COALESCE(m.weekday_morning_departures, 0) AS weekday_morning_departures
FROM station_summary s
LEFT JOIN morning m ON m.station = s.station
WHERE s.pct_deficit_days >= 70
  AND s.total_movements >= 2000
ORDER BY s.month_net_flow ASC
LIMIT 10;

Running query in 'DuckDBPyConnection'

station,month_net_flow,net_flow_pct_of_volume,pct_deficit_days,total_movements,weekday_morning_departures
E 39 St & Lexington Ave,-640,-4.5,83.3,14338,1561
E 33 St & Park Ave,-613,-7.9,100.0,7717,845
Willoughby Ave & Hall St,-493,-4.6,83.3,10693,1334
E 39 St & 2 Ave,-492,-4.5,88.9,11044,1094
E 51 St & 2 Ave,-478,-4.4,86.7,10974,1102
W 45 St & 8 Ave,-348,-2.1,76.7,16464,1332
E 35 St & Madison Ave,-345,-3.5,76.7,9859,879
6 Ave & W 34 St,-323,-1.9,83.3,17385,1453
46 St & Queens Blvd,-321,-9.7,86.7,3299,226
E 47 St & 2 Ave,-300,-1.9,76.7,15712,1626


> **Recommendation:** the 5 stations with the biggest month deficit that also pass the filters are E 39 St & Lexington Ave (-640), E 33 St & Park Ave (-613), Willoughby Ave & Hall St (-493), E 39 St & 2 Ave (-492) and E 51 St & 2 Ave (-478).
> - They are in deficit on 83.3% to 100% of the days, lose 4.4% to 7.9% of their volume on net, and have 845 to 1,561 weekday departures between 6h and 9h.
> - Why these: they lose bikes over the whole month, on most days, and the demand peak is in the morning commute (Q2).
> - Limitations: one month of data, no stock or dock capacity data, and about 0.3% of trips have no station and are not counted.

## Final answer

**Where and when should the company focus overnight rebalancing?**

> The company should prioritize overnight rebalancing at E 39 St & Lexington Ave, E 33 St & Park Ave, Willoughby Ave & Hall St, E 39 St & 2 Ave and E 51 St & 2 Ave, so they are stocked before the weekday morning peak (8h). These stations lose between 478 and 640 bikes over the month, are in deficit on at least 83% of the days, and have between 845 and 1,561 weekday departures between 6h and 9h. Members drive most of the commute-like routes, while leisure round trips are concentrated near Central Park. Results should be read with caution: the analysis covers one month, the data has no station stock or dock capacity, and 62 station names are linked to more than one id.